# Entity-Relationship Modelling

## Overview
An **Entity-Relationship (ER) model** is a conceptual blueprint of a database, created before writing SQL. It identifies what things exist (entities), what facts describe them (attributes), and how they relate (relationships).

| Concept | Definition | Healthcare example |
|---|---|---|
| **Entity** | A distinct object the database tracks | Patient, Provider, Encounter |
| **Attribute** | A property of an entity | patient_id, last_name, dob |
| **Relationship** | An association between entities | Patient has Encounters |
| **Cardinality** | How many instances relate | One-to-many, many-to-many |
| **Primary key** | Attribute(s) uniquely identifying each instance | patient_id |
| **Foreign key** | Attribute referencing another entity's PK | encounter.patient_id |

**Crow's foot cardinality notation:**
```
Patient  ||--o{  Encounter    one patient has zero or more encounters
Encounter }o--|| Provider     each encounter has exactly one provider
Provider ||--o|  Department   each provider belongs to at most one dept
Patient  }o--o{  Diagnosis    patients and diagnoses: many-to-many
```
- `||` exactly one (mandatory)  |  `o|` zero or one (optional)
- `o{` zero or many             |  `|{` one or many (mandatory many)

**Conceptual -> Logical -> Physical progression:**
1. **Conceptual** -- entities and relationships, no data types
2. **Logical** -- adds attributes, PKs, FKs, cardinality
3. **Physical** -- adds data types, constraints, indexes, engine-specific details

**Resolving many-to-many:** A bridge table holds FKs from both parent tables plus any attributes of the relationship itself.

---

In [ ]:
import sqlite3
import pandas as pd
conn = sqlite3.connect(":memory:")
conn.executescript("""CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT NOT NULL, last_name TEXT NOT NULL,dob TEXT NOT NULL, sex TEXT CHECK(sex IN ('M','F','O')), province TEXT, active INTEGER DEFAULT 1);CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT NOT NULL UNIQUE, building TEXT, floor_no INTEGER);CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT NOT NULL, specialty TEXT,licence_no TEXT UNIQUE, province TEXT, dept_id INTEGER REFERENCES departments(dept_id),manager_id INTEGER REFERENCES providers(provider_id));CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT NOT NULL, category TEXT NOT NULL);CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id),provider_id INTEGER NOT NULL REFERENCES providers(provider_id),dept_id INTEGER REFERENCES departments(dept_id),enc_date TEXT NOT NULL, cost_cad REAL, admitted INTEGER DEFAULT 0 CHECK(admitted IN (0,1)));CREATE TABLE encounter_diagnoses (enc_id INTEGER NOT NULL REFERENCES encounters(enc_id),diag_code TEXT NOT NULL REFERENCES diagnoses(diag_code),diag_rank INTEGER DEFAULT 1, confirmed INTEGER DEFAULT 1,PRIMARY KEY (enc_id, diag_code));CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER NOT NULL REFERENCES patients(patient_id),test_name TEXT NOT NULL, result_val REAL, units TEXT, collected TEXT NOT NULL, flagged INTEGER DEFAULT 0);INSERT INTO departments VALUES (1,'Emergency','Tower A',1),(2,'Cardiology','Tower B',2),(3,'Oncology','Tower C',3),(4,'Family Medicine','Clinic',1),(5,'Orthopaedics','Tower A',2),(6,'Radiology','Tower B',3),(7,'ICU','Tower C',NULL);INSERT INTO providers VALUES(10,'Dr. Sarah MacNeil','Cardiology','MC-1001','NB',2,NULL),(11,'Dr. James Wong','Oncology','MC-1002','BC',3,10),(12,'Dr. Fatima Osei','Family Medicine','MC-1003','ON',4,10),(13,'Dr. Ethan Larson','Orthopaedics','MC-1004','QC',5,10),(14,'Dr. Priya Sharma','Emergency','MC-1005','AB',1,10),(15,'Dr. Lucas Petit','Radiology','MC-1006','QC',6,11);INSERT INTO patients VALUES(1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),(3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),(5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),(7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),(9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),(11,'Diana','Flores','2000-02-14','F','NB',1);INSERT INTO diagnoses VALUES('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');INSERT INTO encounters VALUES(1,1,10,2,'2023-01-15',3200.00,1),(2,2,14,1,'2023-02-20',1850.00,1),(3,3,12,4,'2023-03-05',120.00,0),(4,4,13,5,'2023-03-18',5500.00,1),(5,5,12,4,'2023-04-02',95.00,0),(6,6,14,1,'2023-05-11',780.00,0),(7,7,10,2,'2023-06-22',2100.00,1),(8,8,12,4,'2023-07-14',80.00,0),(9,1,14,1,'2023-08-30',410.00,0),(10,3,12,4,'2023-09-12',110.00,0),(11,9,10,2,'2023-10-01',1750.00,1),(12,10,14,1,'2023-11-03',2200.00,1),(13,2,10,2,'2023-11-20',2900.00,1),(14,6,12,4,'2023-12-01',115.00,0);INSERT INTO encounter_diagnoses VALUES(1,'I25.1',1,1),(1,'I10',2,1),(2,'J18.9',1,1),(3,'Z00.0',1,1),(4,'M17.1',1,1),(5,'J06.9',1,1),(6,'R07.9',1,1),(7,'I10',1,1),(7,'I48.0',2,1),(9,'R55',1,1),(10,'Z00.0',1,1),(11,'I48.0',1,1),(12,'S52.5',1,1),(13,'I25.1',1,1),(14,'Z00.0',1,1);INSERT INTO lab_results VALUES(1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),(3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),(5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),(7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),(9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),(11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);""")
conn.commit()
print("Healthcare schema ready.")
for t in ["patients","providers","departments","diagnoses","encounters","encounter_diagnoses","lab_results"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t}: {n} rows")

---
## Entities and primary keys

In [ ]:
print("=== Entity primary keys ===")
summary = [
    ("patients",    "patient_id INTEGER",   "Surrogate -- immune to real-world changes"),
    ("providers",   "provider_id INTEGER",  "Surrogate; licence_no also UNIQUE (candidate key)"),
    ("departments", "dept_id INTEGER",       "Surrogate; dept_name also UNIQUE"),
    ("diagnoses",   "diag_code TEXT",        "Natural PK -- ICD-10 codes are stable and external"),
]
df = pd.DataFrame(summary, columns=["Entity", "Primary Key", "Notes"])
print(df.to_string(index=False))
print()
print("Surrogate keys: generated integers, independent of business meaning.")
print("Natural keys:   real-world identifiers (ICD-10, ISBN) -- use only when stable.")

print()
print("=== Table structures ===")
for t in ["patients", "providers", "departments", "diagnoses"]:
    cols = conn.execute(f"PRAGMA table_info({t})").fetchall()
    print(f"  {t}: {[c[1] for c in cols]}")


---
## Relationships and foreign keys

In [ ]:
print("=== One-to-many relationships ===")
rels = [
    ("patients",    "encounters",   "encounters.patient_id   -> patients.patient_id"),
    ("providers",   "encounters",   "encounters.provider_id  -> providers.provider_id"),
    ("departments", "encounters",   "encounters.dept_id      -> departments.dept_id"),
    ("patients",    "lab_results",  "lab_results.patient_id  -> patients.patient_id"),
    ("departments", "providers",    "providers.dept_id       -> departments.dept_id"),
]
for parent, child, fk in rels:
    print(f"  {parent} ||--o{{ {child}   FK: {fk}")

print()
print("=== Self-referencing relationship ===")
print("  providers ||--o{ providers   FK: providers.manager_id -> providers.provider_id")
q = """
SELECT p.provider_id, p.full_name, p.specialty,
       COALESCE(m.full_name, '(no manager)') AS manager
FROM   providers AS p
LEFT JOIN providers AS m ON p.manager_id = m.provider_id
ORDER BY p.provider_id
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Many-to-many: bridge table

In [ ]:
print("=== Many-to-many resolution via encounter_diagnoses ===")
print()
print("Direct many-to-many (cannot implement):")
print("  patients }o--o{ diagnoses")
print()
print("Resolved with bridge table:")
print("  patients  ||--o{ encounters")
print("  encounters }o--|| providers")
print("  encounters ||--o{ encounter_diagnoses")
print("  diagnoses  ||--o{ encounter_diagnoses")
print()
print("Bridge table encounter_diagnoses structure:")
cols = conn.execute("PRAGMA table_info(encounter_diagnoses)").fetchall()
for c in cols:
    pk_flag = " (PK)" if c[5] else ""
    nn_flag = " NOT NULL" if c[3] else ""
    print(f"  {c[1]:20s} {c[2]:10s}{pk_flag}{nn_flag}")
print()
print("=== Sample: encounters with all their diagnoses ===")
q = """
SELECT e.enc_id, e.enc_date, p.last_name AS patient,
       GROUP_CONCAT(ed.diag_code || ' (' || d.category || ')', ', ') AS diagnoses
FROM   encounters AS e
JOIN   patients   AS p  ON e.patient_id = p.patient_id
JOIN   encounter_diagnoses AS ed ON e.enc_id = ed.enc_id
JOIN   diagnoses  AS d  ON ed.diag_code = d.diag_code
GROUP BY e.enc_id
ORDER BY e.enc_date
"""
print(pd.read_sql(q, conn).to_string(index=False))
conn.close()


---
## Common Pitfalls

**1. Skipping the ERD and going straight to SQL**
Starting with CREATE TABLE before drawing the entities and relationships leads to missed relationships, wrong cardinality, and schema changes that require costly migrations. Sketch the ERD first -- even informally.

**2. Modelling many-to-many without a bridge table**
Storing multiple values in one column (e.g. `diag_codes = 'I25.1,I10'`) breaks first normal form and makes querying require string parsing. Always resolve many-to-many with a bridge table with a composite primary key.

**3. Confusing cardinality with participation (mandatory vs optional)**
`||` (exactly one, mandatory) vs `o|` (zero or one, optional) determines whether a FK is NOT NULL or nullable. A provider must have a specialty (NOT NULL) vs a provider may have a manager (nullable FK). Crow's foot captures this distinction explicitly.

**4. Using natural keys for entities that can change**
Using `email` or `licence_no` as a PK breaks when the value changes. Surrogate integer keys are immune to real-world changes. Reserve natural keys for stable, externally-defined codes like ICD-10 diagnosis codes.

**5. Forgetting relationship attributes belong on the bridge table**
`encounter_diagnoses` needs `diag_rank` (primary vs secondary diagnosis) and `confirmed` -- attributes of the relationship, not of either parent. Storing them on `patients` or `diagnoses` is incorrect. They belong only on the bridge.

**6. Conflating conceptual and physical design decisions**
Questions like "should this be VARCHAR(50) or TEXT?" belong in physical design. During logical modelling, focus on entities, attributes, and relationships. Premature physical decisions create scope creep and distract from getting the data model right.


---
*sql_methods_library - Samantha McGarrigle*